In [ ]:
# %pip install numpy matplotlib pillow

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

categories = ["wires", "cars", "cats", "dogs", "house"]

images = []
labels = []

for category in categories:
    folder = Path("photos") / category

    files = sorted(
        p for p in folder.iterdir()
        if p.suffix.lower() in [".jpg", ".jpeg", ".png"]
    )

    for file in files[:5]:
        with Image.open(file) as img:
            img = img.convert("RGB")
            img = img.resize((32, 32))
            images.append(np.array(img))

        labels.append(category)

images = np.stack(images)

print("Dataset shape:", images.shape)
print("Data type:", images.dtype)


In [ ]:

def PlotSample(index):
    plt.figure(figsize=(3, 3))
    plt.imshow(images[index])
    plt.title(f"Sample {index}: {labels[index]}")
    plt.axis("off")
    plt.show()

PlotSample(0)


In [ ]:

n = images.shape[0]

X = images.reshape(n, -1).astype(np.float64)

X = X / 255.0

print("Shape:", X.shape)
print("Minimum:", X.min())
print("Maximum:", X.max())


In [ ]:

mean = X.mean(axis=0)

X_centered = X - mean

print("Mean before:", X.mean())
print("Mean after:", X_centered.mean())


In [ ]:

cov = np.cov(X_centered, rowvar=False)

# SVD
U, S, Vt = np.linalg.svd(cov)

# Variance along each principal direction
variances = S**2 / (n - 1)

# Regularization to avoid division by zero
epsilon = 0.1

# ZCA whitening
X_zca = (
    (X_centered @ Vt.T)
    / np.sqrt(variances + epsilon)
) @ Vt

print("Whitened shape:", X_zca.shape)


In [ ]:

def display_rescaled(sample):
    # Convert numerical values to 0–1
    # for visualization only
    sample = sample.reshape(32, 32, 3)
    low = sample.min()
    high = sample.max()
    return (sample - low) / (high - low + 1e-12)

index = 14

fig, axes = plt.subplots(1, 2, figsize=(13, 3))

axes[0].imshow(images[index])
axes[0].set_title("Original")

axes[1].imshow(display_rescaled(X_zca[index]))
axes[1].set_title("Whitened")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()
